# BI Auto Interpretation Quickstart (Colab + T4)

This notebook generates BI SFT data and chat-format training text for Qwen3.5 small models.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

!nvidia-smi

In [ ]:
import json
import random
from datetime import datetime, timedelta
from pathlib import Path

random.seed(42)
dataset_path = "/content/bi_sft.json"
text_jsonl_path = "/content/bi_sft_text.jsonl"

rows = []
base_date = datetime(2026, 3, 7)
for i in range(200):
    d = (base_date - timedelta(days=i)).strftime("%Y-%m-%d")
    gmv = round(random.uniform(800, 3500), 2)
    conv = round(random.uniform(1.2, 4.5), 2)
    refund = round(random.uniform(1.0, 12.0), 2)
    roas = round(random.uniform(1.5, 6.5), 2)
    gmv_mom = round(random.uniform(-0.25, 0.25), 3)

    report = {
        "date": d,
        "metrics": {
            "gmv_wan": gmv,
            "conversion_pct": conv,
            "refund_pct": refund,
            "roas": roas,
            "gmv_mom": gmv_mom,
        },
    }

    summary = (
        f"{d}日报：GMV {gmv}万元，转化率 {conv}%，退款率 {refund}%，ROAS {roas}，"
        f"GMV环比 {gmv_mom * 100:.1f}%。"
    )
    advice = "建议优先排查异常指标对应渠道，并补充次日跟踪动作。"

    rows.append({
        "instruction": "请解读这份BI日报并给出结论与建议",
        "input": json.dumps(report, ensure_ascii=False),
        "output": f"{summary} {advice}",
    })

Path(dataset_path).write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", dataset_path, "samples:", len(rows))

def to_chat_text(sample):
    return (
        "<|im_start|>system\n"
        "你是企业BI分析助手，输出要简洁、准确、可执行。\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{sample['instruction']}\n"
        f"{sample['input']}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{sample['output']}\n"
        "<|im_end|>"
    )

with open(text_jsonl_path, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps({"text": to_chat_text(row)}, ensure_ascii=False) + "\n")

print("saved:", text_jsonl_path)
print("preview:")
print(to_chat_text(rows[0])[:300])

In [ ]:
def route_model(result_json):
    required = ["summary", "risks", "actions", "confidence"]
    if any(k not in result_json for k in required):
        return "qwen3.5-2B"
    if float(result_json.get("confidence", 0)) < 0.75:
        return "qwen3.5-2B"
    return "qwen3.5-0.8B"

sample_a = {"summary": "ok", "risks": [], "actions": ["keep"], "confidence": 0.82}
sample_b = {"summary": "low confidence", "risks": ["drop"], "actions": ["check"], "confidence": 0.43}

print("router sample A ->", route_model(sample_a))
print("router sample B ->", route_model(sample_b))